# Figure: correlation-function estimator scaling

Loads `results/corrfunc/scaling.json` (`bench/corrfunc/scaling.py`). Never recomputes.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

_here = Path.cwd()
_candidates = [
    _here / "results" / "corrfunc",
    _here.parents[1] / "results" / "corrfunc",
]
RESULTS = next((c for c in _candidates if c.exists()), _candidates[0])
FIGDIR = RESULTS.parents[1] / "examples" / "differentiable_paper" / "figures"
FIGDIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})
PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3"]
BASELINE = "#8C8C8C"


In [ ]:
data = json.loads((RESULTS / "scaling.json").read_text())
meta = data["metadata"]
records = data["records"]
ns = [r["n"] for r in records]
acc = [r["accumulate"]["min_s"] * 1e3 for r in records]
vg = [r["value_and_grad"]["min_s"] * 1e3 for r in records]
bf = [(r["n"], r["brute_force_s"] * 1e3) for r in records if "brute_force_s" in r]
print(f"device={meta['device_kind']} jax={meta['jax_version']}")


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.2), constrained_layout=True)
ax.plot(ns, acc, "o-", color=PALETTE[0], label="soft counts (forward)")
ax.plot(ns, vg, "s-", color=PALETTE[1], label="soft counts (fwd+grad)")
if bf:
    ax.plot([b[0] for b in bf], [b[1] for b in bf], "^--", color=BASELINE,
            label="brute force O(N^2)")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("N particles"); ax.set_ylabel("time [ms]")
ax.set_title(f"Differentiable pair-count estimator scaling ({meta['device_kind']})")
ax.legend()
out = FIGDIR / "fig_corrfunc_scaling.pdf"
fig.savefig(out); fig.savefig(out.with_suffix(".png"))
print("wrote", out)
